# DLFactor 全市场训练 (Google Colab)
## 步骤
1. 上传 data.arrow 到你的 Google Drive 根目录
2. 上传 train.py 到 Google Drive 根目录  
3. 运行本 Notebook
4. 下载 models/dl_v4/ 到本地

In [ ]:
# 1. 挂载 Google Drive + 安装依赖
from google.colab import drive
drive.mount('/content/drive')

!pip install pyarrow scikit-learn scipy onnx 2>/dev/null

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# 2. 复制文件到 Colab 本地 (加速读取)
import shutil, os

drive_root = '/content/drive/MyDrive'
local_data = '/content/data.arrow'
local_script = '/content/train.py'

# 复制 train.py
shutil.copy(f'{drive_root}/train.py', local_script)

# 复制 data.arrow (6.5GB, 约 2-3 分钟)
print('正在复制 data.arrow... (6.5GB, 需 2-3 分钟)')
shutil.copy(f'{drive_root}/data.arrow', local_data)
print(f'文件大小: {os.path.getsize(local_data)/1024/1024/1024:.1f} GB')

import sys; sys.path.insert(0, '/content')
from train import *

In [ ]:
# 3. 训练 (500只小批量验证, 约 5 分钟)
import time, json, os
os.makedirs('/content/models/dl_v4', exist_ok=True)

class Args: pass
args = Args()
args.hidden_units = 48
args.hidden_layers = 2
args.dropout = 0.40
args.lr = 0.0005
args.epochs = 5
args.batch_size = 2048
args.output = '/content/models/dl_v4'

# 用全量数据
t0 = time.time()
(X_train, y_train, X_val, y_val, X_test, y_test), _ = load_data(
    local_data, None, FEATURE_FIELDS, LOOKBACK, PRED_HORIZON)
print(f'加载: {time.time()-t0:.0f}s')

model, scaler, nf = train(args, X_train, y_train, X_val, y_val, X_test, y_test)
export_onnx(model, scaler, nf, LOOKBACK, args.output, FEATURE_FIELDS)

In [ ]:
# 4. 全量训练 (30 epoch, 约 8 分钟)
args.epochs = 30
model, scaler, nf = train(args, X_train, y_train, X_val, y_val, X_test, y_test)
export_onnx(model, scaler, nf, LOOKBACK, args.output, FEATURE_FIELDS)

In [ ]:
# 5. 回传模型到 Google Drive
model_dir = '/content/models/dl_v4'
drive_model = f'{drive_root}/models/dl_v4'
shutil.copytree(model_dir, drive_model, dirs_exist_ok=True)

!ls -lh {model_dir}/
print(f'\n已保存到 Google Drive: models/dl_v4/')
print('从 Google Drive 下载到本地, 部署到 bin/Release/models/dl_v4/')